### This EDA Answers the Following Questions

3. **How is the Kaggle spectrogram calculated?**  

In [7]:
import pandas as pd
import numpy as np  
from scipy.signal import spectrogram
from scipy.ndimage import zoom

In [3]:
# Q2
# This is the first eeg data in the df, I want the confirm that maximum offset is 40.0, and given each eeg sub id is 50 sec and total of 90 sec is right
eeg_data_example_2 = pd.read_parquet(f"../Data/train_eegs/1628180742.parquet")
eeg_spec_example_2 = pd.read_parquet(f"../Data/train_spectrograms/353733.parquet")

eeg_data_example_2.shape, eeg_spec_example_2.shape

((18000, 20), (320, 401))

In [ ]:
# Q3 The cell try to validate the calculation of kaggle spectrogram
# This includes:
# 1. to see wether the eeg data can be converted to kaggle spectrogram

# LL Spec = ( spec(Fp1 - F7) + spec(F7 - T3) + spec(T3 - T5) + spec(T5 - O1) )/4
# LP Spec = ( spec(Fp1 - F3) + spec(F3 - C3) + spec(C3 - P3) + spec(P3 - O1) )/4
# RP Spec = ( spec(Fp2 - F4) + spec(F4 - C4) + spec(C4 - P4) + spec(P4 - O2) )/4
# RL Spec = ( spec(Fp2 - F8) + spec(F8 - T4) + spec(T4 - T6) + spec(T6 - O2) )/4

In [ ]:
# This is version 1 code to calulate the spectrogram 

# Function to compute average spectrogram for a path
def compute_avg_spec(df, path, fs):
    specs = []
    for ch1, ch2 in path:
        diff_signal = df[ch1].values - df[ch2].values
        f, t, Sxx = spectrogram(diff_signal, fs=fs, nperseg=256, noverlap=128)
        specs.append(Sxx)
    avg_spec = np.mean(specs, axis=0)
    return avg_spec  # shape: [freq_bins, time_steps]

# Generate all 4 spectrograms
spectrograms = {}
for region, path in paths.items():
    spec = compute_avg_spec(eeg_data_example_2, path, fs)
    spectrograms[region] = spec  # shape ≈ [129, ~140] depending on nperseg/step

# OPTIONAL: Stack or resize to [320, 401] if needed (e.g., via interpolation or zero-padding)
# Example: Resize using scipy.ndimage.zoom
from scipy.ndimage import zoom
target_shape = (320, 401)
for region in spectrograms:
    orig = spectrograms[region]
    zoom_factors = (target_shape[0] / orig.shape[0], target_shape[1] / orig.shape[1])
    spectrograms[region] = zoom(orig, zoom=zoom_factors)


In [23]:
spectrograms["LL"][0]

array([ 1.08561623e+00,  1.02214193e+00,  8.79734039e-01,  7.30401933e-01,
        6.29444659e-01,  5.82718313e-01,  5.87035835e-01,  6.50941133e-01,
        8.08940709e-01,  1.09792233e+00,  1.47265482e+00,  1.75033176e+00,
        1.73965907e+00,  1.42356586e+00,  1.00787926e+00,  7.06403136e-01,
        5.86615860e-01,  5.72410524e-01,  5.86252630e-01,  6.17192566e-01,
        7.04377830e-01,  8.77996743e-01,  1.06676459e+00,  1.14115179e+00,
        9.98365760e-01,  7.30955541e-01,  5.16215801e-01,  4.91497010e-01,
        5.84729612e-01,  6.56011164e-01,  6.04110479e-01,  4.76034790e-01,
        3.54248911e-01,  2.98872411e-01,  3.06646228e-01,  3.63117367e-01,
        4.47978437e-01,  5.28479159e-01,  5.70682287e-01,  5.67668319e-01,
        5.56051433e-01,  5.75539708e-01,  6.36390388e-01,  7.12588489e-01,
        7.74815619e-01,  7.71502197e-01,  6.30054295e-01,  2.96257645e-01,
       -2.15818342e-02,  7.49435499e-02,  9.11535263e-01,  2.08429766e+00,
        2.78670311e+00,  

In [24]:
eeg_spec_example_2

,time,LL_0.59,LL_0.78,LL_0.98,LL_1.17,LL_1.37,LL_1.56,LL_1.76,LL_1.95,LL_2.15,...,RP_18.16,RP_18.36,RP_18.55,RP_18.75,RP_18.95,RP_19.14,RP_19.34,RP_19.53,RP_19.73,RP_19.92
0,1,4.26,10.98,9.05,13.65,11.49,8.930000,18.840000,19.26,19.240000,...,0.31,0.17,0.28,0.19,0.24,0.27,0.29,0.16,0.22,0.19
1,3,2.65,3.97,12.18,13.26,14.21,13.230000,9.650000,8.11,11.280000,...,0.15,0.13,0.14,0.24,0.24,0.36,0.35,0.31,0.36,0.40
2,5,4.18,4.53,8.77,14.26,13.36,16.559999,19.219999,17.51,22.650000,...,0.29,0.21,0.16,0.25,0.28,0.28,0.34,0.48,0.44,0.48
3,7,2.41,3.21,4.92,8.07,5.97,12.420000,10.820000,14.96,21.809999,...,0.33,0.51,0.49,0.64,0.58,0.42,0.32,0.31,0.32,0.33
4,9,2.29,2.44,2.77,4.62,5.39,7.080000,9.840000,12.27,14.410000,...,0.44,0.38,0.48,0.63,0.45,0.45,0.49,0.33,0.31,0.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,631,6.36,6.59,6.60,7.30,4.48,8.400000,13.420000,13.85,16.010000,...,0.14,0.05,0.06,0.04,0.04,0.04,0.05,0.05,0.08,0.11
316,633,4.90,8.80,8.22,5.83,10.21,10.580000,10.250000,13.68,19.549999,...,0.16,0.08,0.06,0.06,0.07,0.04,0.06,0.09,0.07,0.08
317,635,6.07,7.85,11.26,9.20,8.18,9.130000,10.450000,15.09,23.020000,...,0.15,0.13,0.13,0.13,0.10,0.08,0.07,0.09,0.17,0.12
318,637,3.41,3.75,4.80,6.45,6.70,7.960000,8.160000,6.97,9.700000,...,0.13,0.11,0.13,0.07,0.11,0.12,0.09,0.16,0.19,0.19


In [8]:
# Sampling frequency
fs = 200

# Define electrode paths
paths = {
    "LL": [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1")],
    "LP": [("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1")],
    "RP": [("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2")],
    "RL": [("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2")]
}

# Function to compute average spectrogram for a path
def compute_avg_spec_with_labels(df, path, fs, region_name):
    specs = []
    for ch1, ch2 in path:
        signal = df[ch1].values - df[ch2].values
        f, t, Sxx = spectrogram(signal, fs=fs, nperseg=256, noverlap=128)
        specs.append(Sxx)
    
    avg_Sxx = np.mean(specs, axis=0)  # shape: [n_freqs, n_time]
    
    # Optional: interpolate to match [320, 401] if needed
    target_shape = (320, 401)
    zoom_factors = (target_shape[0] / avg_Sxx.shape[0], target_shape[1] / avg_Sxx.shape[1])
    avg_Sxx_resized = zoom(avg_Sxx, zoom=zoom_factors)

    # Construct DataFrame with frequency-labeled columns
    f_resized = np.linspace(f[0], f[-1], target_shape[0])
    time_index = np.arange(target_shape[1])

    df_spec = pd.DataFrame(
        avg_Sxx_resized.T,  # shape: [n_time, n_freq]
        columns=[f"{region_name}_{np.round(freq, 2)}" for freq in f_resized]
    )
    df_spec.insert(0, "time", time_index)
    return df_spec


# Compute for all 4 regions and merge
spec_dfs = [compute_avg_spec_with_labels(eeg_data_example_2, path, fs, region) for region, path in paths.items()]
full_spec_df = pd.concat(spec_dfs, axis=1)

# Drop duplicate 'time' columns (keep the first one)
full_spec_df = full_spec_df.loc[:, ~full_spec_df.columns.duplicated()]

# full_spec_df now looks like your screenshot: 320 rows × (4*freqs + 1) columns
full_spec_df.head()

,time,LL_0.0,LL_0.31,LL_0.63,LL_0.94,LL_1.25,LL_1.57,LL_1.88,LL_2.19,LL_2.51,...,RL_97.18,RL_97.49,RL_97.81,RL_98.12,RL_98.43,RL_98.75,RL_99.06,RL_99.37,RL_99.69,RL_100.0
0,0,1.085616,17.108709,42.953987,46.674576,27.131308,13.371137,25.729332,46.462048,49.853680,...,0.019672,0.017868,0.015875,0.019267,0.042964,0.092191,0.129636,0.109296,0.043715,0.005190
1,1,1.022142,15.084702,38.743782,45.301144,32.335381,21.212126,27.339882,39.585270,41.186676,...,0.023200,0.023514,0.019191,0.017836,0.040791,0.097242,0.142662,0.121339,0.046632,0.002390
2,2,0.879734,11.418964,31.287127,43.403187,42.734497,36.053734,29.507792,24.967127,23.181551,...,0.029233,0.034004,0.026149,0.016603,0.037927,0.106420,0.165840,0.144527,0.056625,0.003845
3,3,0.730402,9.719782,28.342419,44.312958,50.511192,44.877869,28.238981,11.623747,7.829070,...,0.030945,0.039661,0.032270,0.020154,0.039851,0.110760,0.175325,0.159454,0.075558,0.024332
4,4,0.629445,12.126780,34.437481,49.755966,50.619759,39.700821,21.492960,5.823009,3.199214,...,0.025013,0.035122,0.034841,0.030841,0.049323,0.104485,0.156245,0.152766,0.101089,0.068343


In [10]:
spec_dfs

[     time    LL_0.0    LL_0.31    LL_0.63    LL_0.94    LL_1.25    LL_1.57  \
 0       0  1.085616  17.108709  42.953987  46.674576  27.131308  13.371137   
 1       1  1.022142  15.084702  38.743782  45.301144  32.335381  21.212126   
 2       2  0.879734  11.418964  31.287127  43.403187  42.734497  36.053734   
 3       3  0.730402   9.719782  28.342419  44.312958  50.511192  44.877869   
 4       4  0.629445  12.126780  34.437481  49.755966  50.619759  39.700821   
 ..    ...       ...        ...        ...        ...        ...        ...   
 396   396  0.143447   2.512141   8.098816  14.619975  19.694689  20.846567   
 397   397  0.298670   4.000998  11.363192  16.930490  18.850409  18.800053   
 398   398  0.484169   5.180544  13.528747  17.252737  15.513682  14.759912   
 399   399  0.636643   5.900867  14.561911  16.392405  11.590239  10.568225   
 400   400  0.699943   6.144773  14.825585  15.786241   9.700077   8.635277   
 
        LL_1.88    LL_2.19    LL_2.51  ...  LL_97.